# Treino e controle por gestos

Este notebook usa as fotos em `src/image/move` e `src/image/stop` para treinar um classificador KNN sobre os landmarks da mão detectados pelo MediaPipe. A classe `move` representa a mão fechada em repouso/parado; a classe `stop` representa a extensão do dedo indicador como gesto de movimento.

Execute as celulas em ordem. A celula final abre a webcam; para encerrar, pressione **F12** em qualquer janela ou **q** com a janela de video selecionada. Se a captura falhar, confira `camera_index` na chamada final.

No macOS, o Terminal/VS Code/Python pode precisar de permissao em **Privacidade e Seguranca > Acessibilidade** para simular teclas e **Monitoramento de Entrada** para reconhecer F12 globalmente. Depois que a webcam abrir, deixe o jogo/emulador em foco para receber a tecla.

O conjunto atual tem poucas fotos e algumas podem ser quase duplicadas. A acuracia leave-one-out e apenas indicativa; para avaliar de verdade, adicione fotos variadas de sessoes diferentes e teste em imagens que nao participaram do treino.

In [1]:
from pathlib import Path

import cv2
import mediapipe as mp
import numpy as np

ROOT = Path.cwd()
DATA_DIR = ROOT / "src" / "image"
CLASSES = ("move", "stop")

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Pasta do dataset nao encontrada: {DATA_DIR}")


def classify_hand_gesture(hand_landmarks):
    points = np.array(
        [[point.x, point.y, point.z] for point in hand_landmarks.landmark],
        dtype=np.float32,
    )
    if points.shape[0] != 21:
        return None

    fingertip_ids = (4, 8, 12, 16, 20)
    fingertip_reach = [
        float(np.linalg.norm(points[tip_id, :2] - points[0, :2]))
        for tip_id in fingertip_ids
    ]

    if max(fingertip_reach) < 0.21 and np.median(fingertip_reach) < 0.16:
        return "mao_fechada"

    index_is_longest = (
        fingertip_reach[1] > 0.18
        and fingertip_reach[1] >= fingertip_reach[2] + 0.03
        and fingertip_reach[1] >= fingertip_reach[3] + 0.03
        and fingertip_reach[1] >= fingertip_reach[4] + 0.03
    )
    if index_is_longest:
        return "indicador"

    return None


def extract_features(hand_landmarks):
    points = np.array(
        [[point.x, point.y, point.z] for point in hand_landmarks.landmark],
        dtype=np.float32,
    )
    wrist = points[0].copy()
    palm_scale = float(np.linalg.norm(points[9, :2] - wrist[:2]))
    if palm_scale < 1e-6:
        return None

    points -= wrist
    points /= palm_scale
    return np.concatenate((points.reshape(-1), wrist[:2], [palm_scale])).astype(
        np.float32
    )


features = []
labels = []
skipped_images = []

with mp.solutions.hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.35,
) as hand_detector:
    for label_id, label in enumerate(CLASSES):
        class_dir = DATA_DIR / label
        image_files = sorted(
            path
            for path in class_dir.iterdir()
            if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
        )
        if not image_files:
            raise ValueError(f"Nenhuma imagem encontrada em {class_dir}")

        for image_path in image_files:
            image = cv2.imread(str(image_path))
            if image is None:
                skipped_images.append((image_path, "arquivo nao pode ser lido"))
                continue

            result = hand_detector.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            if not result.multi_hand_landmarks:
                skipped_images.append((image_path, "mao nao detectada"))
                continue

            feature = extract_features(result.multi_hand_landmarks[0])
            if feature is None:
                skipped_images.append((image_path, "landmarks invalidos"))
                continue

            features.append(feature)
            labels.append(label_id)

features = np.asarray(features, dtype=np.float32)
labels = np.asarray(labels, dtype=np.int32)

if not len(features):
    raise ValueError("Nao foi possivel extrair landmarks de nenhuma imagem.")

for label_id, label in enumerate(CLASSES):
    count = int(np.sum(labels == label_id))
    print(f"{label}: {count} imagens validas")
print(f"Formato das caracteristicas: {features.shape}")
print(f"Imagens ignoradas: {len(skipped_images)}")


I0000 00:00:1791644972.113291  137231 gl_context.cc:357] GL version: 2.1 (2.1 ATI-7.0.25), renderer: AMD Radeon Pro 560X OpenGL Engine
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1791644972.133686  137341 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791644972.165686  137341 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/Users/francismaralvesmartinsjunior/vision_game/.venv/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


move: 17 imagens validas
stop: 11 imagens validas
Formato das caracteristicas: (28, 66)
Imagens ignoradas: 0


In [2]:
K_NEIGHBORS = 3
MIN_CONFIDENCE = 0.80
MODEL_PATH = DATA_DIR / "gesture_model.npz"


def predict_knn(sample, reference_features, reference_labels, k=K_NEIGHBORS):
    distances = np.linalg.norm(reference_features - sample, axis=1)
    nearest_indices = np.argsort(distances)[: min(k, len(reference_labels))]
    votes = np.bincount(
        reference_labels[nearest_indices], minlength=len(CLASSES)
    )
    predicted_label = int(np.argmax(votes))
    confidence = float(votes[predicted_label] / len(nearest_indices))
    return predicted_label, confidence


leave_one_out_predictions = []
for index, sample in enumerate(features):
    training_mask = np.arange(len(labels)) != index
    prediction, _ = predict_knn(
        sample,
        features[training_mask],
        labels[training_mask],
    )
    leave_one_out_predictions.append(prediction)

confusion = np.zeros((len(CLASSES), len(CLASSES)), dtype=np.int32)
for actual, predicted in zip(labels, leave_one_out_predictions):
    confusion[actual, predicted] += 1

accuracy = float(np.mean(labels == leave_one_out_predictions))
print(f"Acuracia leave-one-out (indicativa): {accuracy:.1%}")
print("Matriz de confusao (linhas=real, colunas=previsto):")
print(confusion)

np.savez_compressed(
    MODEL_PATH,
    features=features,
    labels=labels,
    classes=np.asarray(CLASSES),
    k=np.asarray(K_NEIGHBORS),
)
print(f"Modelo salvo em: {MODEL_PATH}")
print(f"Limiar de confianca para ativar a tecla: {MIN_CONFIDENCE:.0%}")


Acuracia leave-one-out (indicativa): 100.0%
Matriz de confusao (linhas=real, colunas=previsto):
[[17  0]
 [ 0 11]]
Modelo salvo em: /Users/francismaralvesmartinsjunior/vision_game/src/image/gesture_model.npz
Limiar de confianca para ativar a tecla: 80%


In [3]:
import threading

from pynput import keyboard as pynput_keyboard


def run_webcam_controller(camera_index=0):
    if not MODEL_PATH.is_file():
        raise FileNotFoundError("Execute primeiro a celula de treinamento do modelo.")

    with np.load(MODEL_PATH, allow_pickle=False) as saved_model:
        model_features = saved_model["features"].astype(np.float32)
        model_labels = saved_model["labels"].astype(np.int32)
        class_names = tuple(str(name) for name in saved_model["classes"].tolist())
        model_k = int(saved_model["k"])

    if "move" not in class_names or "stop" not in class_names:
        raise ValueError("O modelo precisa conter as classes 'move' e 'stop'.")

    controller = pynput_keyboard.Controller()
    move_key = pynput_keyboard.Key.right
    stop_event = threading.Event()
    key_is_down = False
    listener = None
    camera = cv2.VideoCapture(camera_index)

    def stop_on_f12(key):
        if key == pynput_keyboard.Key.f12:
            stop_event.set()
            return False
        return None

    try:
        if not camera.isOpened():
            raise RuntimeError(
                f"Nao foi possivel abrir a camera de indice {camera_index}."
            )

        listener = pynput_keyboard.Listener(on_press=stop_on_f12)
        listener.start()

        with mp.solutions.hands.Hands(
            static_image_mode=False,
            max_num_hands=1,
            model_complexity=0,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5,
        ) as hand_detector:
            while not stop_event.is_set():
                success, frame = camera.read()
                if not success:
                    raise RuntimeError("A camera parou de enviar imagens.")

                result = hand_detector.process(
                    cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                )
                detected_class = None
                confidence = 0.0

                if result.multi_hand_landmarks:
                    hand_landmarks = result.multi_hand_landmarks[0]
                    gesture_name = classify_hand_gesture(hand_landmarks)
                    if gesture_name == "mao_fechada":
                        detected_class = "stop"
                        confidence = 0.96
                    elif gesture_name == "indicador":
                        detected_class = "move"
                        confidence = 0.96
                    else:
                        feature = extract_features(hand_landmarks)
                        if feature is not None:
                            class_id, confidence = predict_knn(
                                feature,
                                model_features,
                                model_labels,
                                k=model_k,
                            )
                            detected_class = (
                                "stop" if class_names[class_id] == "move" else "move"
                            )
                    mp.solutions.drawing_utils.draw_landmarks(
                        frame,
                        hand_landmarks,
                        mp.solutions.hands.HAND_CONNECTIONS,
                    )

                should_hold_key = (
                    detected_class == "move" and confidence >= MIN_CONFIDENCE
                )
                if should_hold_key and not key_is_down:
                    controller.press(move_key)
                    key_is_down = True
                elif not should_hold_key and key_is_down:
                    controller.release(move_key)
                    key_is_down = False

                if detected_class is None:
                    status = "Sem mao detectada - tecla solta"
                elif confidence < MIN_CONFIDENCE:
                    status = f"Incerto ({confidence:.0%}) - tecla solta"
                elif detected_class == "move":
                    status = f"Dedo indicador (movimento, {confidence:.0%})"
                else:
                    status = f"Mao fechada (parada, {confidence:.0%})"

                cv2.putText(
                    frame,
                    status,
                    (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.8,
                    (0, 220, 0) if should_hold_key else (0, 0, 255),
                    2,
                )
                cv2.imshow("Controle por gestos | F12 ou q para sair", frame)
                if cv2.waitKey(1) & 0xFF in (ord("q"), 27):
                    break
    finally:
        if key_is_down:
            controller.release(move_key)
        camera.release()
        cv2.destroyAllWindows()
        if listener is not None:
            listener.stop()
            if listener.is_alive():
                listener.join(timeout=1)

    print("Controle encerrado; tecla e camera liberadas.")


In [ ]:
run_webcam_controller(camera_index=0)


2026-10-10 12:09:34.511 Python[19767:137231] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.
I0000 00:00:1791644975.439118  137231 gl_context.cc:357] GL version: 2.1 (2.1 ATI-7.0.25), renderer: AMD Radeon Pro 560X OpenGL Engine
W0000 00:00:1791644975.450717  137786 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791644975.460523  137786 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
This process is not trusted! Input event monitoring will not be possible until it is added to accessibility clients.


Controle encerrado; tecla e camera liberadas.


: 